In [203]:
import os
import random
import warnings
import numpy as np
import pandas as pd

In [204]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report,  classification_report,confusion_matrix,accuracy_score,balanced_accuracy_score,f1_score,recall_score,precision_score
from sklearn.utils.class_weight import compute_class_weight

In [205]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

In [206]:
train=pd.read_csv('Train_ULAK.csv')

In [207]:
test=pd.read_csv('Test_ULAK.csv')

In [208]:
train.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,68855579,10,6,1038,11595,346,0,103.800000,167.133879,4344,0,1932.5,1754.831473,1.834710e+02,0.232370,4.590372e+06,1.760000e+07,68300000,2,68700000,7633007.889,2.270000e+07,68300000,2,602465,1.204930e+05,1.934046e+05,444104,49,0,0,0,0,328,200,0.145232,0.087139,0,4344,743.117647,1341.078106,1.798490e+06,0,0,0,0,1,0,0,0,0,789.562500,103.800000,1932.5,328,0,0,0,0,0,0,10,1038,6,11595,251,235,3,32,998.0,0.0,998,998,68300000.0,0.0,68300000,68300000,DoS Hulk
1,53,196,2,2,70,174,35,35,35.000000,0.000000,87,87,87.0,0.000000,1.244898e+06,20408.163270,6.533333e+01,1.079645e+02,190,3,3,3.000,0.000000e+00,3,3,3,3.000000e+00,0.000000e+00,3,3,0,0,0,0,64,64,10204.081630,10204.081630,35,87,55.800000,28.481573,8.112000e+02,0,0,0,0,0,0,0,0,1,69.750000,35.000000,87.0,64,0,0,0,0,0,0,2,70,2,174,-1,-1,1,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,123,118,2,2,96,96,48,48,48.000000,0.000000,48,48,48.0,0.000000,1.627119e+06,33898.305080,3.933333e+01,6.206717e+01,111,3,3,3.000,0.000000e+00,3,3,4,4.000000e+00,0.000000e+00,4,4,0,0,0,0,40,40,16949.152540,16949.152540,48,48,48.000000,0.000000,0.000000e+00,0,0,0,0,0,0,0,0,1,60.000000,48.000000,48.0,40,0,0,0,0,0,0,2,96,2,96,-1,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,80,295657,7,10,1114,15841,1084,0,159.142857,407.829796,2920,0,1584.1,792.273304,5.734686e+04,57.499061,1.847856e+04,5.298594e+04,209299,1,295464,49244.000,8.134591e+04,209299,192,262576,2.917511e+04,8.701429e+04,261213,2,0,0,0,0,152,208,23.676084,33.822977,0,2920,941.944444,968.551646,9.380923e+05,0,0,0,1,0,0,0,0,1,997.352941,159.142857,1584.1,152,0,0,0,0,0,0,7,1114,10,15841,8192,11,6,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,80,15705351,7,5,407,452,377,0,58.142857,140.620563,220,0,90.4,118.333427,5.469473e+01,0.764071,1.427759e+06,3.221493e+06,9767208,82,15700000,2617558.500,4.134537e+06,9767208,375,10100000,2.532037e+06,4.846203e+06,9799947,1261,0,0,0,0,152,108,0.445708,0.318363,0,377,66.076923,123.295351,1.520174e+04,0,0,0,1,0,0,0,0,0,71.583333,58.142857,90.4,152,0,0,0,0,0,0,7,407,5,452,8192,16384,6,20,360718.0,0.0,360718,360718,9767208.0,0.0,9767208,9767208,BENIGN


In [209]:
train[' Label']=train[' Label'].str.strip()

In [210]:
train.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,68855579,10,6,1038,11595,346,0,103.800000,167.133879,4344,0,1932.5,1754.831473,1.834710e+02,0.232370,4.590372e+06,1.760000e+07,68300000,2,68700000,7633007.889,2.270000e+07,68300000,2,602465,1.204930e+05,1.934046e+05,444104,49,0,0,0,0,328,200,0.145232,0.087139,0,4344,743.117647,1341.078106,1.798490e+06,0,0,0,0,1,0,0,0,0,789.562500,103.800000,1932.5,328,0,0,0,0,0,0,10,1038,6,11595,251,235,3,32,998.0,0.0,998,998,68300000.0,0.0,68300000,68300000,DoS Hulk
1,53,196,2,2,70,174,35,35,35.000000,0.000000,87,87,87.0,0.000000,1.244898e+06,20408.163270,6.533333e+01,1.079645e+02,190,3,3,3.000,0.000000e+00,3,3,3,3.000000e+00,0.000000e+00,3,3,0,0,0,0,64,64,10204.081630,10204.081630,35,87,55.800000,28.481573,8.112000e+02,0,0,0,0,0,0,0,0,1,69.750000,35.000000,87.0,64,0,0,0,0,0,0,2,70,2,174,-1,-1,1,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,123,118,2,2,96,96,48,48,48.000000,0.000000,48,48,48.0,0.000000,1.627119e+06,33898.305080,3.933333e+01,6.206717e+01,111,3,3,3.000,0.000000e+00,3,3,4,4.000000e+00,0.000000e+00,4,4,0,0,0,0,40,40,16949.152540,16949.152540,48,48,48.000000,0.000000,0.000000e+00,0,0,0,0,0,0,0,0,1,60.000000,48.000000,48.0,40,0,0,0,0,0,0,2,96,2,96,-1,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,80,295657,7,10,1114,15841,1084,0,159.142857,407.829796,2920,0,1584.1,792.273304,5.734686e+04,57.499061,1.847856e+04,5.298594e+04,209299,1,295464,49244.000,8.134591e+04,209299,192,262576,2.917511e+04,8.701429e+04,261213,2,0,0,0,0,152,208,23.676084,33.822977,0,2920,941.944444,968.551646,9.380923e+05,0,0,0,1,0,0,0,0,1,997.352941,159.142857,1584.1,152,0,0,0,0,0,0,7,1114,10,15841,8192,11,6,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,80,15705351,7,5,407,452,377,0,58.142857,140.620563,220,0,90.4,118.333427,5.469473e+01,0.764071,1.427759e+06,3.221493e+06,9767208,82,15700000,2617558.500,4.134537e+06,9767208,375,10100000,2.532037e+06,4.846203e+06,9799947,1261,0,0,0,0,152,108,0.445708,0.318363,0,377,66.076923,123.295351,1.520174e+04,0,0,0,1,0,0,0,0,0,71.583333,58.142857,90.4,152,0,0,0,0,0,0,7,407,5,452,8192,16384,6,20,360718.0,0.0,360718,360718,9767208.0,0.0,9767208,9767208,BENIGN


In [211]:
pd.set_option('display.max_columns', None)

In [212]:
train.columns=train.columns.str.strip()

In [213]:
train.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,68855579,10,6,1038,11595,346,0,103.800000,167.133879,4344,0,1932.5,1754.831473,1.834710e+02,0.232370,4.590372e+06,1.760000e+07,68300000,2,68700000,7633007.889,2.270000e+07,68300000,2,602465,1.204930e+05,1.934046e+05,444104,49,0,0,0,0,328,200,0.145232,0.087139,0,4344,743.117647,1341.078106,1.798490e+06,0,0,0,0,1,0,0,0,0,789.562500,103.800000,1932.5,328,0,0,0,0,0,0,10,1038,6,11595,251,235,3,32,998.0,0.0,998,998,68300000.0,0.0,68300000,68300000,DoS Hulk
1,53,196,2,2,70,174,35,35,35.000000,0.000000,87,87,87.0,0.000000,1.244898e+06,20408.163270,6.533333e+01,1.079645e+02,190,3,3,3.000,0.000000e+00,3,3,3,3.000000e+00,0.000000e+00,3,3,0,0,0,0,64,64,10204.081630,10204.081630,35,87,55.800000,28.481573,8.112000e+02,0,0,0,0,0,0,0,0,1,69.750000,35.000000,87.0,64,0,0,0,0,0,0,2,70,2,174,-1,-1,1,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,123,118,2,2,96,96,48,48,48.000000,0.000000,48,48,48.0,0.000000,1.627119e+06,33898.305080,3.933333e+01,6.206717e+01,111,3,3,3.000,0.000000e+00,3,3,4,4.000000e+00,0.000000e+00,4,4,0,0,0,0,40,40,16949.152540,16949.152540,48,48,48.000000,0.000000,0.000000e+00,0,0,0,0,0,0,0,0,1,60.000000,48.000000,48.0,40,0,0,0,0,0,0,2,96,2,96,-1,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,80,295657,7,10,1114,15841,1084,0,159.142857,407.829796,2920,0,1584.1,792.273304,5.734686e+04,57.499061,1.847856e+04,5.298594e+04,209299,1,295464,49244.000,8.134591e+04,209299,192,262576,2.917511e+04,8.701429e+04,261213,2,0,0,0,0,152,208,23.676084,33.822977,0,2920,941.944444,968.551646,9.380923e+05,0,0,0,1,0,0,0,0,1,997.352941,159.142857,1584.1,152,0,0,0,0,0,0,7,1114,10,15841,8192,11,6,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,80,15705351,7,5,407,452,377,0,58.142857,140.620563,220,0,90.4,118.333427,5.469473e+01,0.764071,1.427759e+06,3.221493e+06,9767208,82,15700000,2617558.500,4.134537e+06,9767208,375,10100000,2.532037e+06,4.846203e+06,9799947,1261,0,0,0,0,152,108,0.445708,0.318363,0,377,66.076923,123.295351,1.520174e+04,0,0,0,1,0,0,0,0,0,71.583333,58.142857,90.4,152,0,0,0,0,0,0,7,407,5,452,8192,16384,6,20,360718.0,0.0,360718,360718,9767208.0,0.0,9767208,9767208,BENIGN


In [214]:
train=train.drop(columns=['Destination Port','Flow Duration','Fwd Packet Length Max','Fwd Packet Length Min','Fwd Packet Length Mean','Fwd Packet Length Std','Bwd Packet Length Max','Bwd Packet Length Min','Bwd Packet Length Mean','Bwd Packet Length Std','Flow Bytes/s','Flow Packets/s'])

In [215]:
train=train.drop(columns=[
    "Flow IAT Mean", "Flow IAT Std", "Flow IAT Max", "Flow IAT Min",
    "Fwd IAT Total", "Fwd IAT Mean", "Fwd IAT Std", "Fwd IAT Max", "Fwd IAT Min",
    "Bwd IAT Total", "Bwd IAT Mean", "Bwd IAT Std", "Bwd IAT Max", "Bwd IAT Min",
    "Fwd PSH Flags", "Bwd PSH Flags", "Fwd URG Flags", "Bwd URG Flags",
    "Fwd Header Length", "Bwd Header Length", "Fwd Packets/s", "Bwd Packets/s",
    "Min Packet Length", "Max Packet Length", "Packet Length Mean", "Packet Length Std", "Packet Length Variance",
    "FIN Flag Count", "SYN Flag Count", "RST Flag Count", "PSH Flag Count", "ACK Flag Count", "URG Flag Count",
    "CWE Flag Count", "ECE Flag Count", "Down/Up Ratio", "Average Packet Size",
    "Avg Fwd Segment Size", "Avg Bwd Segment Size", "Fwd Header Length.1",
    "Fwd Avg Bytes/Bulk", "Fwd Avg Packets/Bulk", "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk", "Bwd Avg Packets/Bulk", "Bwd Avg Bulk Rate",
    "Subflow Fwd Packets", "Subflow Fwd Bytes", "Subflow Bwd Packets", "Subflow Bwd Bytes",
    "Init_Win_bytes_forward", "Init_Win_bytes_backward", "act_data_pkt_fwd", "min_seg_size_forward"
])

In [216]:
train.head()

,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,10,6,1038,11595,998.0,0.0,998,998,68300000.0,0.0,68300000,68300000,DoS Hulk
1,2,2,70,174,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,2,2,96,96,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,7,10,1114,15841,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,7,5,407,452,360718.0,0.0,360718,360718,9767208.0,0.0,9767208,9767208,BENIGN


In [217]:
train.shape

(724031, 13)

In [218]:
train['Label'].value_counts()

Label
BENIGN              581639
DoS Hulk             58790
PortScan             40800
DDoS                 32679
DoS GoldenEye         2635
FTP-Patator           1988
SSH-Patator           1558
DoS slowloris         1481
DoS Slowhttptest      1398
Bot                    487
webattack2             382
webattack1             178
Infiltration             9
webattack3               6
Heartbleed               1
Name: count, dtype: int64

In [219]:
train=train[~train['Label'].isin(['Bot', 'Web Attack-Brute Force', 'Web Attack-XSS','Infiltration','Web Attack-Sql Injection','Heartbleed'])]

In [220]:
train['Label'].value_counts()

Label
BENIGN              581639
DoS Hulk             58790
PortScan             40800
DDoS                 32679
DoS GoldenEye         2635
FTP-Patator           1988
SSH-Patator           1558
DoS slowloris         1481
DoS Slowhttptest      1398
webattack2             382
webattack1             178
webattack3               6
Name: count, dtype: int64

In [221]:
train=train[~train['Label'].isin(['Web Attack ï¿½ Brute Force', 'Web Attack ï¿½ XSS','Web Attack ï¿½ Sql Injection'])]

In [222]:
train['Label'].value_counts()

Label
BENIGN              581639
DoS Hulk             58790
PortScan             40800
DDoS                 32679
DoS GoldenEye         2635
FTP-Patator           1988
SSH-Patator           1558
DoS slowloris         1481
DoS Slowhttptest      1398
webattack2             382
webattack1             178
webattack3               6
Name: count, dtype: int64

In [223]:
train=train[~train['Label'].isin(['webattack1','webattack2','webattack3'])]

In [224]:
test.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,998,2,0,0,0,0,0,0.0,0.0,0,0,0.0,0.0,0.000000e+00,2004.008016,9.980000e+02,0.000000e+00,998,998,998,998.0,0.0,998,998,0,0.0,0.0,0,0,0,0,0,0,64,0,2004.008016,0.000000,0,0,0.0,0.000000,0.0,0,0,0,0,1,0,0,0,0,0.00,0.0,0.0,64,0,0,0,0,0,0,2,0,0,0,251,-1,0,32,0.0,0.0,0,0,0.0,0.0,0,0,DoS Hulk
1,80,63111103,7,0,0,0,0,0,0.0,0.0,0,0,0.0,0.0,0.000000e+00,0.110916,1.050000e+07,1.190000e+07,32100000,999612,63100000,10500000.0,11900000.0,32100000,999612,0,0.0,0.0,0,0,0,0,0,0,280,0,0.110916,0.000000,0,0,0.0,0.000000,0.0,0,0,0,1,0,0,0,0,0,0.00,0.0,0.0,280,0,0,0,0,0,0,7,0,0,0,29200,-1,0,40,7015565.0,0.0,7015565,7015565,18700000.0,12200000.0,32100000,8015910,DoS Slowhttptest
2,53,202,2,2,98,130,49,49,49.0,0.0,65,65,65.0,0.0,1.128713e+06,19801.980200,6.733333e+01,3.348632e+01,106,48,48,48.0,0.0,48,48,48,48.0,0.0,48,48,0,0,0,0,40,40,9900.990099,9900.990099,49,65,55.4,8.763561,76.8,0,0,0,0,0,0,0,0,1,69.25,49.0,65.0,40,0,0,0,0,0,0,2,98,2,130,-1,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,80,3,2,0,0,0,0,0,0.0,0.0,0,0,0.0,0.0,0.000000e+00,666666.666700,3.000000e+00,0.000000e+00,3,3,3,3.0,0.0,3,3,0,0.0,0.0,0,0,0,0,0,0,64,0,666666.666700,0.000000,0,0,0.0,0.000000,0.0,0,0,0,0,1,0,0,0,0,0.00,0.0,0.0,64,0,0,0,0,0,0,2,0,0,0,274,-1,0,32,0.0,0.0,0,0,0.0,0.0,0,0,DoS Hulk
4,7200,37,1,1,0,6,0,0,0.0,0.0,6,6,6.0,0.0,1.621622e+05,54054.054050,3.700000e+01,0.000000e+00,37,37,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,40,20,27027.027030,27027.027030,0,6,2.0,3.464102,12.0,0,0,0,1,0,0,0,0,1,3.00,0.0,6.0,40,0,0,0,0,0,0,1,0,1,6,29200,0,0,40,0.0,0.0,0,0,0.0,0.0,0,0,PortScan


In [225]:
test.columns=test.columns.str.strip()

In [226]:
test=test.drop(columns=['Destination Port','Flow Duration','Fwd Packet Length Max','Fwd Packet Length Min','Fwd Packet Length Mean','Fwd Packet Length Std','Bwd Packet Length Max','Bwd Packet Length Min','Bwd Packet Length Mean','Bwd Packet Length Std','Flow Bytes/s','Flow Packets/s'])

In [227]:
test=test.drop(columns=[
    "Flow IAT Mean", "Flow IAT Std", "Flow IAT Max", "Flow IAT Min",
    "Fwd IAT Total", "Fwd IAT Mean", "Fwd IAT Std", "Fwd IAT Max", "Fwd IAT Min",
    "Bwd IAT Total", "Bwd IAT Mean", "Bwd IAT Std", "Bwd IAT Max", "Bwd IAT Min",
    "Fwd PSH Flags", "Bwd PSH Flags", "Fwd URG Flags", "Bwd URG Flags",
    "Fwd Header Length", "Bwd Header Length", "Fwd Packets/s", "Bwd Packets/s",
    "Min Packet Length", "Max Packet Length", "Packet Length Mean", "Packet Length Std", "Packet Length Variance",
    "FIN Flag Count", "SYN Flag Count", "RST Flag Count", "PSH Flag Count", "ACK Flag Count", "URG Flag Count",
    "CWE Flag Count", "ECE Flag Count", "Down/Up Ratio", "Average Packet Size",
    "Avg Fwd Segment Size", "Avg Bwd Segment Size", "Fwd Header Length.1",
    "Fwd Avg Bytes/Bulk", "Fwd Avg Packets/Bulk", "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk", "Bwd Avg Packets/Bulk", "Bwd Avg Bulk Rate",
    "Subflow Fwd Packets", "Subflow Fwd Bytes", "Subflow Bwd Packets", "Subflow Bwd Bytes",
    "Init_Win_bytes_forward", "Init_Win_bytes_backward", "act_data_pkt_fwd", "min_seg_size_forward"
])

In [228]:
test['Label'].value_counts()

Label
BENIGN              411203
DoS Hulk             41801
PortScan             28751
DDoS                 23160
DoS GoldenEye         1861
FTP-Patator           1436
SSH-Patator           1067
DoS slowloris         1048
DoS Slowhttptest       994
Bot                    355
webattack2             272
webattack1             117
Infiltration             6
webattack3               4
Heartbleed               2
Name: count, dtype: int64

In [229]:
test=test[~test['Label'].isin(['Bot', 'Web Attack-Brute Force', 'Web Attack-XSS','Infiltration','Web Attack-Sql Injection','Heartbleed'])]

In [230]:
test['Label'].value_counts()

Label
BENIGN              411203
DoS Hulk             41801
PortScan             28751
DDoS                 23160
DoS GoldenEye         1861
FTP-Patator           1436
SSH-Patator           1067
DoS slowloris         1048
DoS Slowhttptest       994
webattack2             272
webattack1             117
webattack3               4
Name: count, dtype: int64

In [231]:
test=test[~test['Label'].isin(['webattack1','webattack2','webattack3'])]

In [232]:
test['Label'].value_counts()

Label
BENIGN              411203
DoS Hulk             41801
PortScan             28751
DDoS                 23160
DoS GoldenEye         1861
FTP-Patator           1436
SSH-Patator           1067
DoS slowloris         1048
DoS Slowhttptest       994
Name: count, dtype: int64

In [233]:
train.isnull().sum()

Total Fwd Packets              0
Total Backward Packets         0
Total Length of Fwd Packets    0
Total Length of Bwd Packets    0
Active Mean                    0
Active Std                     0
Active Max                     0
Active Min                     0
Idle Mean                      0
Idle Std                       0
Idle Max                       0
Idle Min                       0
Label                          0
dtype: int64

In [234]:
test.isnull().sum()

Total Fwd Packets              0
Total Backward Packets         0
Total Length of Fwd Packets    0
Total Length of Bwd Packets    0
Active Mean                    0
Active Std                     0
Active Max                     0
Active Min                     0
Idle Mean                      0
Idle Std                       0
Idle Max                       0
Idle Min                       0
Label                          0
dtype: int64

In [235]:
from sklearn.model_selection  import train_test_split

In [236]:
le=LabelEncoder()

In [237]:
train['Label']=le.fit_transform(train['Label'])
test['Label']=le.transform(test['Label'])

In [238]:
test['Label'].value_counts()

Label
0    411203
3     41801
7     28751
1     23160
2      1861
6      1436
8      1067
5      1048
4       994
Name: count, dtype: int64

In [239]:
X1=train.drop(columns=['Label'])
Y1=train['Label']
X2=test.drop(columns=['Label'])
Y2=test['Label']

In [240]:
from imblearn.under_sampling import RandomUnderSampler

In [241]:
plan={0:30000,3:30000,7:30000,1:30000}

In [242]:
plan2={0:30000,3:30000}

In [243]:
rus=RandomUnderSampler(sampling_strategy=plan,random_state=42)

In [244]:
X1under,Y1under=rus.fit_resample(X1,Y1)

In [245]:
rus=RandomUnderSampler(sampling_strategy=plan2,random_state=42)

In [246]:
X2under,Y2under=rus.fit_resample(X2,Y2)

In [247]:
X1_train,X1_test,Y1_train,Y1_test=train_test_split(X1,Y1,test_size=0.2,random_state=42)

In [248]:
scaler=StandardScaler()

In [249]:
X1_train=scaler.fit_transform(X1_train)
X1_test=scaler.transform(X1_test)

In [250]:
from imblearn.over_sampling import SMOTE

In [251]:
smote=SMOTE(random_state=42)

In [252]:
X1over,Y1over=smote.fit_resample(X1_train,Y1_train)

In [253]:
from xgboost import XGBClassifier
     

xgb=XGBClassifier()
     

estimators=[('xgb',xgb)]
     

from sklearn.pipeline import Pipeline
     

pipe=Pipeline(steps=estimators)
     

pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('xgb', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None


In [254]:
from skopt import BayesSearchCV
     

from skopt.space import Real,Categorical,Integer
     

search_space = {
    'max_depth': Integer(3, 8),
    'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
    'n_estimators': Integer(100, 500),
    'subsample': Real(0.6, 1.0),
    'colsample_bytree': Real(0.6, 1.0),
    'reg_alpha': Real(0.0, 5.0),
    'reg_lambda': Real(0.1, 5.0),
    'gamma': Real(0.0, 5.0)
}

In [255]:
opt=BayesSearchCV(pipe,search_space,n_iter=10,cv=3,scoring='roc_auc',random_state=8)
     

opt = BayesSearchCV(
    xgb,
    search_spaces=search_space,
    n_iter=10,
    cv=3
)

In [256]:
opt.fit(X1over,Y1over)

,estimator,"XGBClassifier...ree=None, ...)"
,search_spaces,"{'colsample_bytree': Real(low=0.6,...m='normalize'), 'gamma': Real(low=0.0,...m='normalize'), 'learning_rate': Real(low=0.01...m='normalize'), 'max_depth': Integer(low=3...m='normalize'), ...}"
,optimizer_kwargs,None
,n_iter,10
,scoring,None
,fit_params,None
,n_jobs,1
,n_points,1
,iid,'deprecated'
,refit,True
,cv,3


In [257]:
Y1pred=opt.predict(X1_test)

In [258]:
classification_report(Y1_test,Y1pred)

'              precision    recall  f1-score   support\n\n           0       1.00      0.90      0.94    116389\n           1       0.99      1.00      1.00      6554\n           2       0.95      0.97      0.96       504\n           3       0.73      0.98      0.84     11730\n           4       0.50      0.97      0.66       268\n           5       0.37      1.00      0.54       306\n           6       0.93      1.00      0.96       372\n           7       0.99      0.99      0.99      8171\n           8       0.04      0.98      0.08       300\n\n    accuracy                           0.91    144594\n   macro avg       0.72      0.98      0.77    144594\nweighted avg       0.97      0.91      0.94    144594\n'

In [259]:
accuracy_score(Y1_test,Y1pred)

0.9135994577921629